# 4QDR.AI Universal Problem Solver — Tool Calling with Ollama + Gemma 4

This cookbook demonstrates how a local open-source LLM (Gemma 4 via Ollama) can
**semantically discover and call** the `gemini_web_chat` tool from our pipeline.

## How it works

1. We register the tool definition (name, description, schema) with Ollama
2. Gemma 4 reads the semantic description and decides if/when to call it
3. If the model requests a tool call, we execute `execute_tool()` from `tool_mode`
4. The result is fed back to the model for its final answer

## Prerequisites

- Ollama installed with `gemma4:e4b` (or any tool-calling-capable model)
- Python packages: `pip install ollama`
- Run from the project root so `tool_mode.py` is importable

In [ ]:
# Install ollama Python package if not already installed
# !pip install ollama

In [ ]:
import json
import sys
import os
sys.path.insert(0, os.getcwd())

import ollama
from tool_mode import get_tool_schema, execute_tool
import asyncio

## Step 1: Inspect the tool schema

Our tool definition includes a rich semantic `description` that tells the LLM **what** the tool does, **when** to use it, and **when not** to. This is what enables Gemma 4 to autonomously decide to call it.

In [ ]:
tool_schema = get_tool_schema()
print(json.dumps(tool_schema, indent=2, ensure_ascii=False))

## Step 2: Convert to Ollama-compatible tool format

Ollama expects the OpenAI-style `tools` format:

In [ ]:
ollama_tool = {
    'type': 'function',
    'function': {
        'name': tool_schema['name'],
        'description': tool_schema['description'],
        'parameters': tool_schema['parameters']
    }
}

print(f"Registered tool: {ollama_tool['function']['name']}")
print(f"Description (first 150 chars): {ollama_tool['function']['description'][:150]}...")

## Step 3: Send a query to Gemma 4 with tool access

The model reads the semantic description and decides autonomously whether to invoke the tool.

In [ ]:
messages = [
    {
        'role': 'user',
        'content': (
            'Research the latest advancements in quantum error correction '
            'and summarize them in 3 bullet points. '
            'Use the gemini_web_chat tool if you need real-time info from Gemini.'
        )
    }
]

response = ollama.chat(
    model='gemma4:e4b',
    messages=messages,
    tools=[ollama_tool]
)

print(f"Model: {response['model']}")
print(f"Role: {response['message']['role']}")
print(f"Content: {response['message']['content']}")

## Step 4: Check if the model wants to call the tool

If `tool_calls` is present, the LLM autonomously decided to use Gemini.

In [ ]:
tool_calls = response['message'].get('tool_calls', [])

if not tool_calls:
    print("Model answered directly without calling the tool.")
    print("Final answer:", response['message']['content'])
else:
    print(f"Model requested {len(tool_calls)} tool call(s):")
    for tc in tool_calls:
        print(f"  - {tc['function']['name']}({tc['function']['arguments']})")

## Step 5: Execute the tool and return results

We run `execute_tool()` with the arguments Gemma 4 chose, then send the structured result back.

In [ ]:
async def run_full_flow():
    """Run the complete LLM → tool call → result → LLM loop."""
    tool_calls = response['message'].get('tool_calls', [])
    
    if not tool_calls:
        print("No tool calls to process.")
        return response['message']['content']
    
    # Append the model's response (with tool_calls) to messages
    messages.append(response['message'])
    
    for tc in tool_calls:
        if tc['function']['name'] != 'gemini_web_chat':
            print(f"Unknown tool: {tc['function']['name']}, skipping")
            continue
        
        arguments = json.loads(tc['function']['arguments'])
        print(f"Executing gemini_web_chat with: prompt={arguments.get('prompt', '')[:60]}...")
        
        # Execute the tool (this opens Playwright, talks to Gemini)
        result = await execute_tool(arguments)
        
        # Add the tool result to messages
        messages.append({
            'role': 'tool',
            'content': json.dumps(result, ensure_ascii=False),
            'name': 'gemini_web_chat'
        })
        print(f"Tool result received: success={result.get('success')}, length={result.get('response_length', 0)}")
    
    # Send the conversation back to the LLM for final synthesis
    final_response = ollama.chat(
        model='gemma4:e4b',
        messages=messages
    )
    
    return final_response['message']['content']


# Run the flow
final_answer = await run_full_flow()

## Step 6: View the final answer

Gemma 4 has now incorporated Gemini's response into its own final answer.

In [ ]:
print("=" * 60)
print("FINAL ANSWER FROM GEMMA 4")
print("=" * 60)
print(final_answer)

## Summary

- The **rich semantic description** in our tool definition enables Gemma 4 to autonomously decide to call `gemini_web_chat`
- The model can choose appropriate parameters (model, tool, prompt) based on your query
- Tool results are fed back for the LLM to synthesize a complete answer
- This same pattern works with any Ollama model that supports tool/function calling (Gemma 4, Qwen 3, Mistral, Llama 3.1+, etc.)